<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/Web_Search_API_Tavily.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Web Search API with LlamaIndex + Tavily

This notebook demonstrates how to use **Tavily Search API** with LlamaIndex agents and tools.

> **Why Tavily?** Google's Custom Search JSON API is closed to new customers as of 2025. Tavily is purpose-built for LLM agents — it scrapes, filters, and returns full page content in a single API call, with a free tier of 1,000 searches/month.

**Get your Tavily API key:**
1. Go to [app.tavily.com](https://app.tavily.com) and sign up
2. Your API key is shown on the dashboard immediately (no credit card required)
3. Add it to Colab Secrets as `TAVILY_API_KEY`

## Install Dependencies

In [1]:
!pip install -q llama-index==0.14.12 openai==2.14.0 \
                llama-index-tools-tavily-research \
                newspaper4k==0.9.4.1 lxml-html-clean==0.4.3 jedi==0.19.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 kB 1.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.2/306.2 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 108.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.9/394.9 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2/121.2 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 30.3 MB/s eta 0:00:00
   ━━━━━━

## Set API Keys

In [2]:
import os

# Set the following API Keys in the Python environment. Will be used later.
# os.environ["OPENAI_API_KEY"] = "[OPENAI_API_KEY]"
# TAVILY_API_KEY = "tvly-YOUR_API_KEY"

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
TAVILY_API_KEY = userdata.get('TAVILY_API_KEY')

## LLM and Embedding Model

In [3]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI

Settings.llm = OpenAI(model="gpt-5-mini", additional_kwargs={'reasoning_effort': 'minimal'})
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

## Using Agents / Tools

A quick example with a custom tool before wiring up web search.

In [4]:
from llama_index.core.agent.workflow import ReActAgent
from llama_index.core.workflow import Context

# Define a sample tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers and returns the result integer"""
    return a * b

# Initialize ReAct agent
agent = ReActAgent(tools=[multiply], verbose=True)

# Create a context to store the conversation history / session state
ctx = Context(agent)

In [5]:
from llama_index.core.agent.workflow import AgentStream, ToolCallResult

handler = agent.run("What is the multiplication of 43 and 45?", ctx=ctx)

async for ev in handler.stream_events():
    if isinstance(ev, AgentStream):
        print(f"{ev.delta}", end="", flush=True)

response = await handler
print(str(response))

[tick] add: AgentWorkflowStartEvent(user_msg='What is the multiplication of 43 and 45?', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
[init_run:0] started from AgentWorkflowStartEvent
[init_run:0] complete with AgentInput
[tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='What is the multiplication of 43 and 45?')])], current_agent_name='Ag...
[setup_agent:0] started from AgentInput
[setup_agent:0] complete with AgentSetup
[tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='What is the multiplication of 43 and 45?')])], current_agent_name='Ag...
[run_agent_step:0] started from AgentSetup
Thought: The current language of the user is: English. I need to use a tool to help me answer the question.
Action: multiply
Action Input: {"a": 43, "b": 45}[run_agent_step:0] complete with Ag

## Define Tavily Search Tool

Tavily is purpose-built for LLM agents. Unlike Google Custom Search (which only returned snippets), Tavily scrapes, filters, and returns **full page content** in a single API call — no additional scraping step needed.

In [6]:
from llama_index.tools.tavily_research import TavilyToolSpec

tool_spec = TavilyToolSpec(api_key=TAVILY_API_KEY)

# Tavily returns full content already — no LoadAndSearchToolSpec wrapper needed
search_tools = tool_spec.to_tool_list()

print(f"Available tools: {[t.metadata.name for t in search_tools]}")

Available tools: ['search', 'extract']


## Create the Agent

In [7]:
from llama_index.core.agent.workflow import FunctionAgent

# System prompt encouraging tool usage
system_prompt = """You are a helpful assistant that can search the web for current information.
When you don't have information about recent events or models released after your knowledge cutoff,
use the available search tools to find accurate, up-to-date information."""

search_agent = FunctionAgent(
    tools=search_tools,
    llm=Settings.llm,
    system_prompt=system_prompt,
    verbose=True
)

ctx = Context(search_agent)

In [8]:
handler = search_agent.run(
    "How many parameters does LLaMA 4 have? List the models with parameters",
    ctx=ctx
)

async for ev in handler.stream_events():
    if isinstance(ev, ToolCallResult):
        print(f"\nCall {ev.tool_name} with {ev.tool_kwargs}\nReturned: {ev.tool_output}")
    if isinstance(ev, AgentStream):
        print(f"{ev.delta}", end="", flush=True)

response = await handler

[tick] add: AgentWorkflowStartEvent(user_msg='How many parameters does LLaMA 4 have? List the mo...', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
[init_run:0] started from AgentWorkflowStartEvent
[init_run:0] complete with AgentInput
[tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='How many parameters does LLaMA 4 have? List the models with parameter...
[setup_agent:0] started from AgentInput
[setup_agent:0] complete with AgentSetup
[tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.SYSTEM: 'system'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text="You are a helpful assistant that can search the web for current i...
[run_agent_step:0] started from AgentSetup
[run_agent_step:0] complete with AgentOutput
[tick] add: AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={'tool_calls': [ChoiceDeltaT

In [9]:
print(f"\nFinal Response: {response}")


Final Response: LLaMA 4 uses a mixture-of-experts (MoE) family. Meta’s public announcement and product pages list three LLaMA 4 variants and give both total (stored) parameter counts and the number of parameters activated per token. The models and their parameter sizes are:

- LLaMA 4 Scout
  - Total parameters: ~109 billion
  - Active parameters per token: ~17 billion
  - Experts: 16

- LLaMA 4 Maverick
  - Total parameters: ~400 billion
  - Active parameters per token: ~17 billion
  - Experts: 128

- LLaMA 4 Behemoth (preview / teacher model; not publicly released)
  - Total parameters: ~2 trillion
  - Active parameters per token: ~288 billion
  - Experts: (reported) 16 (with shared expert arrangement)

Notes:
- “Total parameters” = all stored weights across all experts. “Active parameters” = the subset of parameters routed/used for a given token (what matters for inference compute/latency).
- Meta has also referenced other internal/unreleased LLaMA 4 variants (e.g., a reasoning mod

In [10]:

print(f"Tool Calls Made: {response.tool_calls}")

Tool Calls Made: [ToolCallResult(tool_name='search', tool_kwargs={'query': "LLaMA 4 models parameter counts list 'LLaMA 4' 'parameter' models sizes", 'max_results': 6}, tool_id='call_tASK6mqhk2uZOhFyQcqVN99M', tool_output=ToolOutput(blocks=[TextBlock(block_type='text', text='multi-GPU setup (e.g., 8×H100), but efficient due to MoE. Outperforms GPT-4, Gemini 2.0, and Claude 3 in many open benchmarks. Fully open for research and commercial use (except in EU). 🔴 LLaMA 4 Behemoth (preview model) A 2 trillion parameter model, with 288B active parameters — the largest LLaMA yet. Still under training — not publicly released, but shown as Meta’s ultra-high-end “teacher” model. Used for distilling knowledge into smaller models like Scout and Maverick. [...] 🔴LLaMA 4 Scout 109B total parameters, but only 17B active per token using Mixture-of-Experts (MoE). Can run on a single high-end GPU (e.g., H100) with int4 quantization. Supports a massive 10 million token context window, ideal for document-

## Using Tools with VectorStoreIndex

For more precise source attribution and deeper content indexing, we can fetch search results, build a `VectorStoreIndex` from the returned documents, and query it with a standard query engine.

Tavily's `.search()` method already returns full page content as `Document` objects — no separate newspaper scraping step required.

### Fetch Search Results

In [15]:
# Tavily returns Documents with full text — newspaper scraping not needed
search_results = tool_spec.search("LLaMA 4 model details and paramters details", max_results=6)

print(f"Found {len(search_results)} results")

pages_content = []
for doc in search_results:
    url = doc.metadata.get("url", "")
    title = doc.metadata.get("title", "")
    if doc.text:
        pages_content.append({"url": url, "title": title, "text": doc.text})

print(f"Fetched content from {len(pages_content)} pages")

Found 6 results
Fetched content from 6 pages


### Build the Index

In [16]:
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import VectorStoreIndex

documents = [
    Document(text=doc["text"], metadata={"title": doc["title"], "url": doc["url"]})
    for doc in pages_content
]

index = VectorStoreIndex.from_documents(
    documents,
    transformations=[SentenceSplitter(chunk_size=512, chunk_overlap=128)],
)

### Query

In [17]:
query_engine = index.as_query_engine()
response = query_engine.query(
    "How many parameters does LLaMA 4 have? List exact sizes of each variant."
)
print(response)

- LLaMA 4 Scout: 17B active parameters, 109B total parameters (16 experts)
- LLaMA 4 Maverick: 17B active parameters, 400B total parameters (128 experts)
- LLaMA 4 Behemoth: 288B active parameters, 2T (2,000B) total parameters (16 experts)


In [18]:
# Show sources
for node in response.source_nodes:
  print(f"Source: {node.metadata['url']}")
  print(f"Score:  {node.score:.4f}")
  print("-" * 40)

Source: https://www.labellerr.com/blog/llama-4/
Score:  0.5767
----------------------------------------
Source: https://bizon-tech.com/blog/llama-4-system-gpu-requirements-running-locally?srsltid=AfmBOooO-wpBcWZ_U5_nYUjfuer83pSvDoyM-YEDTM92YeoK0pPJ5SZI
Score:  0.5651
----------------------------------------
